In [1]:

#################################################################################################################################################################################################################################################################################
#CONEXION AL ORACLE

import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
from decouple import config

oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()

base2 = pd.read_sql_query(f"SELECT * FROM mbtac10", con=connection2)

connection2.close()



In [2]:
base2

,tipacccod,tipaccdes
0,01,NINGUNO
1,02,COMUN
2,03,DE TRABAJO
3,21,DE TRANSITO C/SOAT
4,22,DE TRANSITO S/SOAT
5,31,VIOLENCIA URBANA
6,32,INTENTO DE HOMICIDIO
7,33,INTENTO DE SUICIDIO
8,99,DESCONOCIDO


In [3]:
a=base2[base2.duplicated(subset=["tipacccod"])]
a

,tipacccod,tipaccdes


In [4]:
base2.columns

Index(['tipacccod', 'tipaccdes'], dtype='object')

In [5]:

#################################################################################################################################################################################################################################################################################
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()



base2.to_sql(name='tmp_mbtac10', con=engine3, if_exists='replace', index=False)

#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL mbtac10 FALSO CON LO OBTENIDO DEL ORACLE

query_count_before = "SELECT COUNT(*) FROM mbtac10"
cant_antes = connection3.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla mbtac10 antes de la inserción: {cant_antes}")



query="""
ALTER TABLE tmp_mbtac10 
ALTER COLUMN tipacccod TYPE CHARACTER (2),
ALTER COLUMN tipaccdes TYPE CHARACTER (20);


UPDATE mbtac10 
SET 

tipaccdes= t.tipaccdes


FROM tmp_mbtac10 t 
WHERE mbtac10.tipacccod = t.tipacccod and mbtac10.tipacccod IS NOT NULL and t.tipacccod IS NOT NULL ;

INSERT INTO mbtac10 
(tipacccod, tipaccdes) 

SELECT 
tipacccod, tipaccdes

FROM tmp_mbtac10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM mbtac10 
    WHERE mbtac10.tipacccod = tmp_mbtac10.tipacccod and mbtac10.tipacccod IS NOT NULL and tmp_mbtac10.tipacccod IS NOT NULL
);


"""

c1= text(query)
connection3.execute(c1)

#BORRAMOS LAS TABLAS
query2="""
DROP TABLE tmp_mbtac10;
DELETE FROM mbtac10 WHERE tipacccod ='';
SELECT COUNT(*) FROM mbtac10;
"""


c2= text(query2)
cursor=connection3.execute(c2)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla mbtac10 después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

base2 = pd.read_sql_query(f"SELECT * FROM mbtac10", con=connection3)

connection3.close()


Cantidad de filas en la tabla mbtac10 antes de la inserción: 9
Cantidad de filas en la tabla mbtac10 después de la inserción: 9
La cantidad de filas insertadas fue de: 0


In [6]:
base2.columns

Index(['tipacccod', 'tipaccdes'], dtype='object')

In [7]:
#################################################################################################################################################################################################################################################################################


#SUBIMOS ESA TABLA ACTUALIZADA COMO TEMPORAL A LA BASE DEL DIM ACTIVIT

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine4 = create_engine(cadena4)
connection4 = engine4.connect()


base2.to_sql(name='tmp_mbtac10', con=engine4, if_exists='replace', index=False)



#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL DIM ACTIVITI FALSO CON LA BASE DE DATOS QUE SUBIMOS

query_count_before = "SELECT COUNT(*) FROM dim_emeacc"
cant_antes = connection4.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla dim_emeacc antes de la inserción: {cant_antes}")


query="""
ALTER TABLE tmp_mbtac10 
ALTER COLUMN tipacccod TYPE CHARACTER (2),
ALTER COLUMN tipaccdes TYPE CHARACTER (20);


INSERT INTO dim_emeacc (cod_acc, des_acc) 
SELECT tipacccod, tipaccdes

FROM tmp_mbtac10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM dim_emeacc 
    WHERE dim_emeacc.cod_acc = tmp_mbtac10.tipacccod
);

DROP TABLE tmp_mbtac10;

SELECT COUNT(*) FROM dim_emeacc;
"""

c1 = text(query)
cursor = connection4.execute(c1)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla dim_emeacc después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

connection4.close()

Cantidad de filas en la tabla dim_emeacc antes de la inserción: 9
Cantidad de filas en la tabla dim_emeacc después de la inserción: 9
La cantidad de filas insertadas fue de: 0
